In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas_datareader import data as pdr
from datetime import datetime, timedelta
import statsmodels.api as sm
from scipy.optimize import minimize
import os
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

In [ ]:
class Utils:
  @staticmethod
  def read_data(ticker, start, end, constrains):
    try:
      t = yf.Ticker(ticker)
      df = t.history(start=start, end=end, interval="1mo")[['Close', 'Volume']]
      df.index = df.index.tz_localize(None).normalize()

      if not Utils.check_constrains(df["Close"].mean(), df["Volume"].mean(), constrains):
        return None, False

      annual_bs = t.balance_sheet

      target_rows = ['Stockholders Equity', 'Share Issued']
      financials = annual_bs.loc[annual_bs.index.intersection(target_rows)].T

      financials.index = pd.to_datetime(financials.index).tz_localize(None).normalize()

      df = pd.merge_asof(
        df.sort_index(),
        financials.sort_index(),
        left_index=True,
        right_index=True,
        direction='backward'
      )

      df["Market_Cap"] = df["Share Issued"] * df["Close"]

      df["BMV"] = df["Stockholders Equity"] / df["Market_Cap"]

      df["Size"] = np.log(df["Market_Cap"])

      df["Momentum"] = df["Close"].pct_change(periods=8)

      df["Returns"] = df["Close"].pct_change()

      df = df[["Size", "BMV", "Momentum", "Returns"]]

      return df, True

    except Exception as e:
      raise Exception(f"Error retrieving data for {ticker}: {e}")


  def check_constrains(close_mean, volume_mean, constrains):
    volume_mean = volume_mean / 1000000

    if constrains is None:
      return True

    if close_mean < constrains["min_price"] or volume_mean < constrains["min_volume"]:
      return False

    return True

  @staticmethod
  def compute_z_scores(x):
    mu = np.mean(x)
    sigma = np.std(x)

    z = (x - mu) / sigma

    return z, sigma

  @staticmethod
  def dot_product(A, B):
    return np.sum(A * B, axis=1).reshape(-1, 1)



In [ ]:
class DataReader:
  def __init__(self, constrains, debug=False):
    self.factors = None
    self.constrains = constrains
    self.market_data = None
    self.debug = debug

  def read_factors(self, reader_params, start, end):
    factors = pdr.DataReader(*reader_params)
    factor_data = factors[0]
    factor_data = factor_data[start:end]
    return factor_data

  def load_marketdata(self, tickers, start, end):
    ticker_data = {}
    for ticker in tickers:
      res, cons_check = Utils.read_data(ticker, start, end, self.constrains)

      if not cons_check:
        print(f"Failed to load: {ticker:^4} | Ticker did not meet the set constrains")
        continue

      ticker_data[ticker] = res
      
      print(f"---------------- {ticker:^4} ----------------")
      if self.debug:
        print(ticker_data[ticker].head(3))

    full_df = pd.concat(ticker_data, axis=1, names=['Ticker', 'Feature'])
    full_df = full_df.swaplevel(0, 1, axis=1).sort_index(axis=1)

    return full_df.dropna()
    
  def generate_data(self, tickers, start, end, reader_params):
    factors = self.read_factors(reader_params, start, end)
    market_data = self.load_marketdata(tickers, start, end)

    return market_data, factors


In [ ]:
class FeatureStore(DataReader):
  def __init__(self, constrains, debug:bool=False):
    super().__init__(constrains, debug=False)
    self.debug = debug

  def compute_scores(self, features, returns):
    def get_corr_i(group, target):
        return group.corrwith(target, axis=1)

    scores = features.groupby(level="Feature", axis=1, group_keys=False).apply(
        lambda x: x.sub(x.mean(axis=1), axis=0).div(x.std(axis=1), axis=0)
    )
    
    forward_returns = returns.shift(-1) 
    IC = features.groupby(level="Feature", axis=1, group_keys=False).apply(
        get_corr_i, target=forward_returns
    )

    return scores.mul(IC, axis=1, level="Feature")

  def construct_factors(self, returns, factors):
    if isinstance(factors.index, pd.PeriodIndex):
        factors.index = factors.index.to_timestamp()

    rf = factors["RF"]

    factor_returns = factors.drop(columns=["RF"])

    common_index = returns.index.intersection(rf.index).intersection(factor_returns.index)
    
    excess_returns = returns.loc[common_index].sub(rf.loc[common_index], axis=0).dropna()
    
    return excess_returns, factor_returns.loc[excess_returns.index]
  
  def load_or_generate_data(self, tickers, start, end, reader_params):
    returns_path = f"returns_{start}_{end}.parquet"
    factors_path = f"factors_{start}_{end}.parquet"
    scores_path = f"scores{start}_{end}.parquet"

    if os.path.exists(returns_path) and os.path.exists(factors_path) and os.path.exists(scores_path) and not self.debug:
      if self.debug:
        print("loading data")

      scores = pd.read_parquet(scores_path)
      returns = pd.read_parquet(returns_path)
      factors = pd.read_parquet(factors_path)

      if self.debug:
        print(scores.head())
        print(returns.head())
        print(factors.head())

    else:
      if self.debug:
        print("generating data")

      market_data, factors_raw = self.generate_data(tickers, start, end, reader_params)
      
      features = market_data[["Size", "Momentum", "BMV"]]
      returns_raw = market_data["Returns"].shift()

      scores = self.compute_scores(features=features, returns=returns_raw)

      returns, factors = self.construct_factors(returns=returns_raw, factors=factors_raw)

      scores.to_parquet(scores_path)
      returns.to_parquet(returns_path)
      factors.to_parquet(factors_path)

    return scores, returns, factors


In [ ]:
class FactorModel:
  def __init__(self, alpha=10.0, err_lambda=0.94, window_size=12, solver="auto", min_var=None):
    self.alpha = alpha
    self.window_size = window_size
    self.solver = solver
    self.lam = err_lambda
    self.epsilon_min = min_var

  def calculate_window_beta(self, x, y):
    model = Ridge(alpha=self.alpha, fit_intercept=True, solver=self.solver)
    model.fit(x, y)
    
    beta = model.coef_
    intercept = model.intercept_
    
    res = (y - model.predict(x))**2
    sigma2 = res.iloc[0]

    for t in range(1, len(res)):
        sigma2 = self.lam * res.iloc[t] + (1 - self.lam) * sigma2

    if self.epsilon_min:
        sigma2 = max(sigma2, self.epsilon_min)

    return (intercept, beta, sigma2)

  def fit(self, X, returns):
    factors_list = X.columns
    tickers = returns.columns
    n_rows = len(returns)

    np_betas = np.full((n_rows, len(tickers) * len(factors_list)), np.nan)
    np_alphas = np.full((n_rows, len(tickers)), np.nan)
    np_idiosyncratic_var = np.full((n_rows, len(tickers)), np.nan)

    cols = pd.MultiIndex.from_product([tickers, factors_list], names=['Ticker', 'Factor'])

    for i, ticker in enumerate(tickers):
        y_series = returns[ticker]
        
        for j in range(self.window_size - 1, n_rows):
            x_window = X.iloc[j-self.window_size+1 : j+1]
            y_window = y_series.iloc[j-self.window_size+1 : j+1]

            if y_window.isnull().any(): continue

            alpha, beta, sigma2 = self.calculate_window_beta(x_window, y_window)

            start_col = i * len(factors_list)
            end_col = start_col + len(factors_list)
            np_betas[j, start_col:end_col] = beta
            
            np_alphas[j, i] = alpha
            np_idiosyncratic_var[j, i] = sigma2

    betas = pd.DataFrame(np_betas, columns=cols, index=returns.index).dropna()
    alphas = pd.DataFrame(np_alphas, columns=tickers, index=returns.index).loc[betas.index]
    idiosyncratic_vars = pd.DataFrame(np_idiosyncratic_var, columns=tickers, index=returns.index).loc[betas.index]

    return alphas, betas, idiosyncratic_vars




In [ ]:
class FactorPremiaModel:
    def __init__(self, alpha=0.5, lam_alpha=0.4):
        self.factors = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"] 
        self.alpha = alpha
        self.lam_alpha = lam_alpha

    def cross_sectional_z(self, df):
        return df.sub(df.mean(axis=1), axis=0).div(df.std(axis=1), axis=0)

    def estimate_lambda(self, factor_returns):

        return factor_returns.mean(axis=0) * self.lam_alpha

    def compute_expected_return(self, betas, scores, factor_returns):
        tickers = scores.columns
        dates = betas.index
        
        lam = self.estimate_lambda(factor_returns)

        systematic_returns_dict = {}
        
        for ticker in tickers:
            beta_i = betas[ticker] 
            systematic_returns_dict[ticker] = beta_i.dot(lam)
            
        systematic_df = pd.DataFrame(systematic_returns_dict, index=dates)


        z_systematic = self.cross_sectional_z(systematic_df)
        z_scores = self.cross_sectional_z(scores)

        expected_returns = (self.alpha * z_scores) + ((1 - self.alpha) * z_systematic)

        return expected_returns.dropna()

In [ ]:
from dataclasses import dataclass

@dataclass
class Constrains:
  investment: float = 1
  max_weight: float = 0.05
  turnover_lim: float = 0.2
  min_weight: float = 0
  beta_dev: float = .1

In [ ]:
class PortfolioObjective:
  def __init__(
      self, lam, betas, factor_premia,
      constrains, factor_penalty, cov,
      turnover_penalty, desired_exp,
      w_prew, use_factor_penalty=True
    ):

    self.lam = lam
    self.betas = betas
    self.factor_premia = factor_premia
    self.cov = cov
    self.constrains = constrains
    self.penalty = factor_penalty
    self.desired_exp = desired_exp
    self.turnover_penalty = turnover_penalty
    self.w_prev = w_prew

  def magnitude(self, x):
    x2_sum = np.sum(x**2)
    return np.sqrt(x2_sum)

  def loss(self, x):
    factor_target = self.penalty * (self.magnitude(self.beta.T - self.desired_exp))**2 if use_factor_penalty else 0
    turnover_penalty = self.turnover_penalty * (self.magnitude(x - self.w_prev))**2
    return x.T * self.factor_premia - self.lam * x  * self.cov * x.T - factor_target - turnover_penalty



In [ ]:
from scipy.optimize import minimize, NonlinearConstraint, LinearConstraint

class Optimizer:
  def __init__(self, obj, lam, betas, factor_premia, cov,
      constrains, factor_penalty,
      turnover_penalty, desired_exp, w_prew):
    self.obj = obj(lam, betas, factor_premia, cov, constrains, factor_penalty, turnover_penalty, desired_exp, w_prew)
    self.factor_opt = None
    self.cons = cons
    self.betas = betas
    self.w_prew = w_prew

  def get_constrains(self, x):
    n = len(x)

    A_sum = np.ones((1, n))
    lb_sum = [1]
    ub_sum = [1]
    w_sum_cons = LinearConstraint(A_sum, lb_sum, ub_sum)


    each_element = lambda x: x
    ub_size = [self.cons.max_weight or 1] * n
    lb_size = [self.cons.min_weight or 0] * n
    min_max_w_cons = NonlinearConstraint(each_element, lb_size, ub_size)

    lb_beta = [1-self.cons.beta_dev]
    ub_beta = [1+self.cons.beta_dev]
    beta_cons = w_sum_cons = LinearConstraint(self.betas, lb_beta, ub_beta)


    A_up = np.hstack([np.eye(n), -np.eye(n)])
    ub_up = w_prev

    A_low = np.hstack([-np.eye(n), -np.eye(n)])
    ub_low = -w_prew

    A_turn = np.hstack([np.zeros(n), np.ones(n)])
    ub_turn = [self.cons.turnover_lim]

    A_final = np.vstack([A_up, A_turn, A_low])
    ub_turnover = np.concatenate(ub_up, ub_turn, up_low)
    lb_turnover = np.full_like(ub_turnover, -np.inf)

    turnover_cons = LinearConstraint(A_final, ub_turnover, lb_turnover)

    return [w_sum_cons, min_max_w_cons, beta_cons, turnover_cons]


  def optimize(self, init = None,
               method: str="SLSQP") -> np.ndarray:
    n = len(self.obj)

    constraints = get_constrains

    results = minimize(
        fun=self.obj.loss,
        x0=init,
        method=method,
        constraints=constraints
    )



    if not results.success:
      raise RuntimeError("Optimization Failed")

    self.factor_opt = results.x
    return self.factor_opt

In [ ]:
class SatelitePortfolioModel:
  def __init__(self, tickers:list, start:str, end:str, constrains:PortfolioObjective,
               reader_params:tuple[str], ridge_alpha:float=10.0, premia_alpha:float=0.5,
               lam_alpha:float=0.3, window:int=12, save_model: bool = False,
               model_path: str|None=None, debug:bool=False, cov_window=6):

    self.dataStore = FeatureStore(
        tickers = tickers,
        start=start,
        end=end,
        cons=constrains,
        reader_params=reader_params,
        debug=debug
    )

    self.cov_window = cov_window

    self.model = FactorModel(alpha=ridge_alpha, window=window)
    self.premia_model = FactorModel(alpha=premia_alpha, lam_alpha=lam_alpha)

  def calculate_covariance(self, ids_risk):
    idx = self.dataStore.returns.index
    n_rows = len(idx)
    n_cols = 1
    cov_empty = np.empty((n_rows, n_cols))
    cov_empty[:] = np.nan
    cov = pd.Series(cov_empty, index=idx)

    factor_cov = self.dataStore.factors.rolling(12).cov()

    for i in range(self.cov_window-1, len(self.dataStore.returns.index)):
      betas_i = betas.iloc[i].unstack(level='Factor').T
      D_i = ids_risk.iloc[i]
      factor_cov = self.dataStore.factors.iloc[i-self.cov_window+1:i+1].cov()

      cov.iloc[i] = betas_i * factor_cov * betas_i.T + D_i

    return cov

  def generate_optim_features(self):
    self.dataStore.load_or_generate_data()
    alphas, betas, ids_risk = self.model.fit(self.dataStore.factors, self.dataStore.returns)
    factor_returns = self.dataStore.factors.pct_change().dropna()
    factor_premia = self.premia_model.fit(betas, self.dataStore.signals, factor_returns)
    cov = self.calculate_covariance(ids_risk)

    return betas, factor_premia, cov

  def optimize_portfolio(self, betas, factor_premia, cov, constrains):
    pass

  def build_portfolios(self, constrains, ):
    betas, factor_premia, cov = self.generate_optim_features()
    self.optimize_portfolio(betas, factor_premia, cov, constrains)


In [ ]:
class CorePortfolioModel:
    def __init__(self, window):
        pass

    def black_litterman_obj_func():
        pass
    
    def optimize(returns, view):
        pass

In [ ]:
class PortfolioConstruction:
  def __init__(self, featureset, save_model: bool = False, model_path:str|None=None):
    pass